# Práctica RAG - Fase 2: Benchmark de Modelos de Embedding

> **Proyecto:** Benchmark de Embeddings y LLMs Locales para RAG  
> **Autores (Alumnos):** Alfred, Ane y Naiara  
> **Hardware de referencia:** GPU NVIDIA A800 (Entorno local con PyTorch y Ollama)  

---

### ¿Qué hacemos en este notebook?
En esta segunda fase evaluamos de forma aislada y empírica la **etapa de recuperación (*retrieval*)**. Queremos saber qué modelo de embedding local encuentra mejor los fragmentos relevantes para responder a nuestras preguntas, **sin usar todavía ningún LLM**.

### Los dos documentos evaluados:
1. **Normativa jurídica:** *Estatuto de los Trabajadores* (BOE consolidado, 420 fragmentos). Artículos formales y muy relacionados entre sí.
2. **Documentación técnica:** *Manual del Administrador de Debian* (1.210 fragmentos). Comandos de terminal, rutas  y términos en inglés dentro del texto.

### Los tres modelos de embedding que comparamos:
- **** (): 0.7B parámetros. Modelo denso moderno de Google DeepMind.
- **** (): 0.11B parámetros. Basado en BERT entrenado específicamente en corpus español.
- **** (): 0.12B parámetros. Modelo multilingüe de Microsoft que usa prefijos obligatorios ( y ).

### ¿Qué métricas medimos?
- **Recall@k (k=1, 3, 5):** ¿El fragmento correcto está entre los $ primeros devueltos? (Recall@5 es nuestro criterio principal).
- **MRR (Mean Reciprocal Rank):** ¿En qué posición aparece el primer fragmento correcto? Cuanto más arriba, mayor puntuación.
- **Coste computacional:** Tiempo de indexado de todos los fragmentos, tamaño del índice en disco (MB) y consumo real de memoria VRAM en GPU.

In [1]:
# Librerías para leer archivos, medir tiempos y cargar los modelos de embeddings.
import json
import time
import os
import torch
from sentence_transformers import SentenceTransformer, util

# Si hay GPU, PyTorch puede usarla para hacer los cálculos más rápido.
# torch.cuda.is_available() devuelve True si PyTorch puede usar una GPU compatible.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo de cómputo en uso: {device}')
# El nombre y la memoria ayudan a saber qué equipo está ejecutando el benchmark.
if device == 'cuda':
    print(f'GPU detectada: {torch.cuda.get_device_name(0)}')
    print(f'VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

/home/g4/reto_1_2-2627-Rag-Benchmark/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo de cómputo en uso: cuda
GPU detectada: NVIDIA A800 40GB Active MIG 3g.20gb
VRAM disponible: 19.62 GB


## 1. Definición de Candidatos y Documentos

In [2]:
# Datos de los modelos que vamos a comparar. Los prefijos se usan en algunos modelos
# para distinguir una pregunta de un texto del documento.
MODELOS = [
    {
        # 'nombre' es el identificador con el que se carga el modelo.
        'nombre': 'google/embeddinggemma-2',
        'alias': 'EmbeddingGemma-2',
        'params': '0.7B',
        'prefijo_query': '',
        'prefijo_doc': '',
        'trust_remote_code': True
    },
    {
        'nombre': 'hiiamsid/sentence_similarity_spanish_es',
        'alias': 'Spanish-BETO',
        'params': '0.11B',
        'prefijo_query': '',
        'prefijo_doc': '',
        'trust_remote_code': False
    },
    {
        'nombre': 'intfloat/multilingual-e5-small',
        'alias': 'Multilingual-E5-Small',
        'params': '0.12B',
        'prefijo_query': 'query: ',
        'prefijo_doc': 'passage: ',
        'trust_remote_code': False
    }
]

# Para cada documento indicamos dónde están sus fragmentos y preguntas.
DOCUMENTOS = [
    {
        'nombre': '1. Normativa (BOE Estatuto de los Trabajadores)',
        'archivo_chunks': 'data/processed/boe_chunks.json',
        'archivo_preguntas': 'preguntas/boe_questions.json'
    },
    {
        'nombre': '2. Técnica (Debian Handbook)',
        'archivo_chunks': 'data/processed/debian_chunks.json',
        'archivo_preguntas': 'preguntas/debian_questions.json'
    }
]

## 2. Funciones de Evaluación (Recall@k y MRR)

Siguiendo los criterios de evaluación del proyecto:
- **Recall@k:** Porcentaje de preguntas donde el fragmento correcto (*ground truth*) aparece en los primeros $k$ resultados ($k=1, 3, 5$).
- **MRR (Mean Reciprocal Rank):** Mide la posición del primer resultado correcto ($1 / \text{rango}$). Premia a los modelos que colocan el fragmento acertado en el puesto 1.

In [3]:
# Compara los fragmentos recuperados con los que sabemos que deberían aparecer.
def calcular_metricas(ranking_resultados, preguntas_con_respuesta):
    # El total solo incluye preguntas que sí tienen una respuesta en el documento.
    total = len(preguntas_con_respuesta)
    if total == 0:
        return 0, 0, 0, 0

    # Estos contadores guardan cuántas preguntas encuentran al menos un fragmento correcto.
    hits_at_1 = 0
    hits_at_3 = 0
    hits_at_5 = 0
    reciprocal_ranks = []

    for q, recuperados in ranking_resultados:
        # Los IDs esperados vienen anotados en las preguntas; los convertimos a conjunto
        # para poder comprobar coincidencias de forma sencilla.
        esperados = set(q['chunks_esperados'])
        if not esperados:
            continue  # Las preguntas sin respuesta se excluyen del cálculo de Recall

        # Nos quedamos con los primeros 1, 3 y 5 resultados del ranking.
        top_1 = set(recuperados[:1])
        top_3 = set(recuperados[:3])
        top_5 = set(recuperados[:5])

        # El símbolo & obtiene los elementos que están en ambos conjuntos.
        if top_1 & esperados:
            hits_at_1 += 1
        if top_3 & esperados:
            hits_at_3 += 1
        if top_5 & esperados:
            hits_at_5 += 1

        # Buscamos la posición del primer fragmento correcto para calcular el MRR.
        primer_rango = None
        for i, chunk_id in enumerate(recuperados, start=1):
            if chunk_id in esperados:
                primer_rango = i
                break

        if primer_rango:
            reciprocal_ranks.append(1.0 / primer_rango)
        else:
            reciprocal_ranks.append(0.0)

    # Recall@k indica en cuántas preguntas aparece una respuesta correcta entre los k primeros.
    # MRR da más puntuación si el primer resultado correcto aparece antes.
    recall_1 = hits_at_1 / total
    recall_3 = hits_at_3 / total
    recall_5 = hits_at_5 / total
    mrr = sum(reciprocal_ranks) / total if reciprocal_ranks else 0.0

    return recall_1, recall_3, recall_5, mrr

## 3. Función de Evaluación por Documento y Medición de GPU

In [4]:
# Ejecuta un modelo con un documento: mide velocidad, memoria y calidad de búsqueda.
def evaluar_modelo_en_documento(modelo_info, documento):
    # Cargamos tanto los textos que se buscarán como las preguntas de evaluación.
    with open(documento['archivo_chunks'], 'r', encoding='utf-8') as f:
        chunks = json.load(f)
    with open(documento['archivo_preguntas'], 'r', encoding='utf-8') as f:
        preguntas = json.load(f)

    # Elegimos GPU cuando está disponible y limpiamos las mediciones anteriores.
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    if dev == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    # 1. Cargar modelo y obtener dimensión
    # SentenceTransformer descarga o carga el modelo indicado y lo prepara en el dispositivo.
    modelo = SentenceTransformer(
        modelo_info['nombre'],
        device=dev,
        trust_remote_code=modelo_info['trust_remote_code']
    )
    # La dimensión es cuántos números contiene el vector de cada texto.
    dimension = modelo.get_embedding_dimension()

    # 2. Medir tiempo de indexado
    # Convertimos cada fragmento en una lista de números (embedding) que representa su contenido.
    textos_docs = [modelo_info['prefijo_doc'] + c['text'] for c in chunks]
    # Guardamos el momento de inicio para medir cuánto tarda en convertir todos los textos.
    start_time = time.time()
    embeddings_docs = modelo.encode(
        textos_docs,
        batch_size=32,
        show_progress_bar=False,
        convert_to_tensor=True,
        normalize_embeddings=True
    )
    # Estimamos el tamaño del índice suponiendo 4 bytes por cada número del vector.
    tiempo_indexado = time.time() - start_time
    tamano_mb = (len(chunks) * dimension * 4) / (1024 * 1024)

    # 3. Vectorizar preguntas y recuperar top-5
    # Estas preguntas se usan para calcular las métricas de recuperación.
    preguntas_con_respuesta = [q for q in preguntas if q['chunks_esperados']]
    ranking_resultados = []

    # Se convierte cada pregunta en vector y se compara con el índice del documento.
    for q in preguntas:
        query_text = modelo_info['prefijo_query'] + q['pregunta']
        query_emb = modelo.encode(query_text, convert_to_tensor=True, normalize_embeddings=True)
        # La similitud coseno compara la pregunta con todos los fragmentos;
        # topk devuelve los índices de los fragmentos con mayor puntuación.
        cos_scores = util.cos_sim(query_emb, embeddings_docs)[0]
        top_indices = torch.topk(cos_scores, k=min(10, len(chunks))).indices.tolist()
        # Traducimos las posiciones encontradas a los IDs que aparecen en los JSON.
        recuperados = [chunks[idx]['id'] for idx in top_indices]
        ranking_resultados.append((q, recuperados))

    # 4. Métricas de búsqueda: comparamos lo recuperado con la respuesta esperada.
    r1, r3, r5, mrr = calcular_metricas(ranking_resultados, preguntas_con_respuesta)

    # 5. Medir VRAM pico
    # En CPU dejamos 0; con GPU consultamos el máximo de memoria usado durante la evaluación.
    vram_gb = 0.0
    if dev == 'cuda':
        vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    # Liberamos objetos grandes antes de evaluar otro modelo para ahorrar memoria.
    del modelo
    del embeddings_docs
    if dev == 'cuda':
        torch.cuda.empty_cache()

    # Devolvemos los datos redondeados para guardarlos y mostrarlos luego.
    return {
        'modelo': modelo_info['alias'],
        'nombre_hf': modelo_info['nombre'],
        'parametros': modelo_info['params'],
        'dimension': dimension,
        'recall_1': round(r1, 3),
        'recall_3': round(r3, 3),
        'recall_5': round(r5, 3),
        'mrr': round(mrr, 3),
        'tiempo_indexado_s': round(tiempo_indexado, 2),
        'tamano_mb': round(tamano_mb, 2),
        'vram_gb': round(vram_gb, 2)
    }

## 4. Ejecución del Benchmark y Generación de Tablas A

In [5]:
# Creamos la carpeta donde se guardará el resultado, si aún no existe.
os.makedirs('resultados', exist_ok=True)
resultados_totales = {}

print('=' * 70)
print('INICIANDO BENCHMARK DE EMBEDDINGS (FASE 2)')
print('=' * 70)

# Probamos todos los modelos con cada uno de los documentos.
for esc in DOCUMENTOS:
    print(f"\n>>> Evaluando Documento: {esc['nombre']}")
    # Abrimos una lista para ir guardando los resultados de cada modelo para este documento.
    resultados_totales[esc['nombre']] = []

    for m_info in MODELOS:
        print(f"  -> Probando modelo: {m_info['alias']}...")
        res = evaluar_modelo_en_documento(m_info, esc)
        # Guardamos el resultado de este modelo antes de pasar al siguiente.
        resultados_totales[esc['nombre']].append(res)
        print(f"     Recall@5: {res['recall_5']} | MRR: {res['mrr']} | Indexado: {res['tiempo_indexado_s']}s | VRAM: {res['vram_gb']} GB")

# Guardamos los resultados para poder consultarlos sin repetir todo el benchmark.
with open('resultados/embeddings_benchmark.json', 'w', encoding='utf-8') as f:
    json.dump(resultados_totales, f, ensure_ascii=False, indent=2)
print('\nResultados guardados correctamente en resultados/embeddings_benchmark.json')

INICIANDO BENCHMARK DE EMBEDDINGS (FASE 2)

>>> Evaluando Documento: 1. Normativa (BOE Estatuto de los Trabajadores)
  -> Probando modelo: EmbeddingGemma-2...


Loading weights:   0%|                                | 0/1376 [00:00<?, ?it/s]

Loading weights: 100%|██████████████████| 1376/1376 [00:00<00:00, 17991.76it/s]

     Recall@5: 1.0 | MRR: 0.865 | Indexado: 2.19s | VRAM: 2.82 GB
  -> Probando modelo: Spanish-BETO...


Loading weights:   0%|                                 | 0/199 [00:00<?, ?it/s]

Loading weights:   7%|█▌                     | 14/199 [00:00<00:01, 133.21it/s]

... (24 outputs left out)

     Recall@5: 0.333 | MRR: 0.261 | Indexado: 14.01s | VRAM: 2.37 GB
  -> Probando modelo: Multilingual-E5-Small...


Loading weights:   0%|                                 | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|████████████████████| 199/199 [00:00<00:00, 12611.30it/s]

     Recall@5: 0.625 | MRR: 0.493 | Indexado: 4.0s | VRAM: 1.8 GB

Resultados guardados correctamente en resultados/embeddings_benchmark.json


## 5. Visualización Formateada de las Tablas A

In [6]:
import pandas as pd
from IPython.display import display, Markdown

# Convertimos los resultados en tablas para compararlos más fácilmente.
for nombre_doc, filas in resultados_totales.items():
    display(Markdown(f"###  {nombre_doc}"))
    
    # DataFrame convierte la lista de diccionarios en una tabla de pandas.
    df = pd.DataFrame(filas)
    # Seleccionamos y ordenamos solo las columnas que queremos enseñar.
    df = df[[
        'modelo', 'parametros', 'dimension', 
        'recall_1', 'recall_3', 'recall_5', 'mrr', 
        'tiempo_indexado_s', 'tamano_mb', 'vram_gb'
    ]]
    # Cambiamos los nombres internos por nombres más claros para la tabla.
    df.columns = [
        'Modelo', 'Parámetros', 'Dimensión', 
        'Recall@1', 'Recall@3', 'Recall@5', 'MRR', 
        'Indexado (s)', 'Índice (MB)', 'VRAM (GB)'
    ]
    
    # Formatear números para lectura clara
    formato = {
        'Recall@1': '{:.3f}',
        'Recall@3': '{:.3f}',
        'Recall@5': '{:.3f}',
        'MRR': '{:.3f}',
        'Indexado (s)': '{:.2f} s',
        'Índice (MB)': '{:.2f} MB',
        'VRAM (GB)': '{:.2f} GB'
    }
    # Aplicamos el formato y quitamos la columna de números de fila.
    tabla_estilizada = df.style.format(formato).hide(axis='index')
    display(tabla_estilizada)


###  1. Normativa (BOE Estatuto de los Trabajadores)

Modelo,Parámetros,Dimensión,Recall@1,Recall@3,Recall@5,MRR,Indexado (s),Índice (MB),VRAM (GB)
EmbeddingGemma-2,0.7B,768,0.750,0.958,1.000,0.865,2.19 s,1.23 MB,2.82 GB
Spanish-BETO,0.11B,768,0.583,0.833,0.917,0.720,2.39 s,1.23 MB,2.13 GB
Multilingual-E5-Small,0.12B,384,0.792,0.833,0.958,0.844,0.97 s,0.62 MB,1.81 GB


###  2. Técnica (Debian Handbook)

Modelo,Parámetros,Dimensión,Recall@1,Recall@3,Recall@5,MRR,Indexado (s),Índice (MB),VRAM (GB)
EmbeddingGemma-2,0.7B,768,0.333,0.542,0.667,0.484,9.73 s,3.54 MB,9.97 GB
Spanish-BETO,0.11B,768,0.208,0.250,0.333,0.261,14.01 s,3.54 MB,2.37 GB
Multilingual-E5-Small,0.12B,384,0.417,0.542,0.625,0.493,4.00 s,1.77 MB,1.80 GB


## 6. Conclusiones y Selección de Ganadores

- **Para el Documento 1 (BOE):** Se elige **`EmbeddingGemma-2`** porque mantiene un **Recall@5 de 1.000 (100%)** y el mayor **MRR (0.865)** incluso al ampliar el conjunto de evaluación a 30 preguntas.
- **Para el Documento 2 (Debian):** Se elige **`Multilingual-E5-Small`** porque obtiene el **mayor Recall@1 (0.417)** y el **mejor MRR (0.493)** (casi igualando a EmbeddingGemma-2 en Recall@5: 0.625 frente a 0.667), con un coste computacional muy inferior (4.00 s de indexado y 1.80 GB de VRAM frente a los 9.97 GB de EmbeddingGemma-2).
- **Decisión global del grupo:** Como elección global para el sistema completo, nos quedamos con **`Multilingual-E5-Small`** de Microsoft. Es el más rápido con diferencia (0.97 s en el BOE y 4.00 s en Debian), genera índices mínimos (0.62 MB y 1.77 MB), logra el mejor Recall@1 en ambos documentos (0.792 en BOE y 0.417 en Debian) y gasta muy poca memoria VRAM (1.80 GB). Aunque en el BOE queda ligeramente por detrás de EmbeddingGemma-2 en Recall@5 (0.958 frente a 1.000), en conjunto es el modelo más consistente, barato y eficiente para producción.